In [16]:
import pandas as pd
import os
import numpy as np
import time
import requests

In [2]:
files = [
    "Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",
    "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
    "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
    "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
    "Resale flat prices based on registration date from Jan-2017 onwards.csv",
]

folder_path = os.path.join("..", "data")

dfs = {}

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dfs[file] = df

    print(f"\n===== {file} =====")
    print(df.head())


===== Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv =====
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06   

   floor_area_sqm      flat_model  lease_commence_date  resale_price  
0            31.0        IMPROVED                 1977          9000  
1            31.0        IMPROVED                 1977          6000  
2            31.0        IMPROVED                 1977          8000  
3            31.0        IMPROVED                 1977          6000  
4            73.0  NEW GENERATION                 1976         47200  

===== Resale Flat Prices (Based on Approval Date), 2000 - Fe

In [3]:
cols_to_check = ["town", "flat_type", "storey_range", "flat_model"]

for name, df in dfs.items():

    print(f"\n================ {name} ================\n")

    for col in cols_to_check:
        if col in df.columns:
            unique_vals = df[col].dropna().unique()

            print(f"\nColumn: {col}")
            print(f"Unique values: {len(unique_vals)}")
            print(sorted(unique_vals))


================ Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv ================


Column: town
Unique values: 26
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'LIM CHU KANG', 'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']

Column: flat_type
Unique values: 7
['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI GENERATION']

Column: storey_range
Unique values: 9
['01 TO 03', '04 TO 06', '07 TO 09', '10 TO 12', '13 TO 15', '16 TO 18', '19 TO 21', '22 TO 24', '25 TO 27']

Column: flat_model
Unique values: 13
['2-ROOM', 'APARTMENT', 'IMPROVED', 'IMPROVED-MAISONETTE', 'MAISONETTE', 'MODEL A', 'MODEL A-MAISONETTE', 'MULTI GENERATION', 'NEW GENERATION', 'PREMIUM APARTMENT', 'SIMPLIFIED', 'STANDARD', 'TERRACE']

====

# Data cleaning

In [4]:
# handle cols month, remaining lease

for name, df in dfs.items():

    # Convert month to datetime
    df["month"] = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")

    # Extract transaction year/month
    df["transaction_year"] = df["month"].dt.year
    df["transaction_month"] = df["month"].dt.month

    # Recalculate remaining lease manually for all tables
    lease_end_year = df["lease_commence_date"] + 99
    total_months = (lease_end_year - df["transaction_year"]) * 12 - (df["transaction_month"] - 1)

    df["remaining_lease_total_months"] = total_months
    df["remaining_lease_years"] = total_months // 12
    df["remaining_lease_months"] = total_months % 12

    # Drop unnecessary old column if it exists
    df.drop(columns=["remaining_lease", "month"], errors="ignore", inplace=True)

    dfs[name] = df

    print(f"\n===== {name} =====")
    print(df.head())


===== Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv =====
         town flat_type block       street_name storey_range  floor_area_sqm  \
0  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12            31.0   
1  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06            31.0   
2  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12            31.0   
3  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09            31.0   
4  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06            73.0   

       flat_model  lease_commence_date  resale_price  transaction_year  \
0        IMPROVED                 1977          9000              1990   
1        IMPROVED                 1977          6000              1990   
2        IMPROVED                 1977          8000              1990   
3        IMPROVED                 1977          6000              1990   
4  NEW GENERATION                 1976         47200              1990   



In [5]:
# handle col flat model

for name, df in dfs.items():

    if "flat_model" in df.columns:
        df["flat_model"] = df["flat_model"].str.upper()

    dfs[name] = df

    print(f"\n===== {name} =====")
    print(df["flat_model"].head())


===== Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv =====
0          IMPROVED
1          IMPROVED
2          IMPROVED
3          IMPROVED
4    NEW GENERATION
Name: flat_model, dtype: object

===== Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv =====
0          IMPROVED
1          IMPROVED
2    NEW GENERATION
3    NEW GENERATION
4    NEW GENERATION
Name: flat_model, dtype: object

===== Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv =====
0          IMPROVED
1          IMPROVED
2    NEW GENERATION
3    NEW GENERATION
4    NEW GENERATION
Name: flat_model, dtype: object

===== Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv =====
0          IMPROVED
1    NEW GENERATION
2    NEW GENERATION
3    NEW GENERATION
4    NEW GENERATION
Name: flat_model, dtype: object

===== Resale flat prices based on registration date from Jan-2017 onwards.csv =====
0          IMPROVED
1    NEW GENERATION
2    NEW GE

In [6]:
for name, df in dfs.items():

    print(f"\n===== {name} =====")
    print(df.columns.tolist())


===== Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv =====
['town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'transaction_year', 'transaction_month', 'remaining_lease_total_months', 'remaining_lease_years', 'remaining_lease_months']

===== Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv =====
['town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'transaction_year', 'transaction_month', 'remaining_lease_total_months', 'remaining_lease_years', 'remaining_lease_months']

===== Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv =====
['town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'transaction_year', 'transaction_month', 'remaining_lease_total_months', 'remaining_lease_years', 'remaining_

In [7]:
combined_df = pd.concat(dfs.values(), ignore_index=True)

print(combined_df.head())
print(combined_df.shape)

         town flat_type block       street_name storey_range  floor_area_sqm  \
0  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12            31.0   
1  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06            31.0   
2  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12            31.0   
3  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09            31.0   
4  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06            73.0   

       flat_model  lease_commence_date  resale_price  transaction_year  \
0        IMPROVED                 1977        9000.0              1990   
1        IMPROVED                 1977        6000.0              1990   
2        IMPROVED                 1977        8000.0              1990   
3        IMPROVED                 1977        6000.0              1990   
4  NEW GENERATION                 1976       47200.0              1990   

   transaction_month  remaining_lease_total_months  remaining_lease_years 

# Feature engineering

In [8]:
combined_df["flat_age"] = combined_df["transaction_year"] - combined_df["lease_commence_date"]
combined_df["price_per_sqm"] = combined_df["resale_price"] / combined_df["floor_area_sqm"]
combined_df["log_price"] = np.log(combined_df["resale_price"])
combined_df["log_price_per_sqm"] = np.log(combined_df["price_per_sqm"])

# mature estates indicator
mature_estates = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT MERAH","BUKIT TIMAH",
    "CENTRAL AREA","GEYLANG","KALLANG/WHAMPOA","MARINE PARADE",
    "QUEENSTOWN","SERANGOON","TOA PAYOH"
]

combined_df["mature_estate"] = combined_df["town"].isin(mature_estates).astype(int)


#flat type mapping
flat_type_map = {
    "1 ROOM":1,
    "2 ROOM":2,
    "3 ROOM":3,
    "4 ROOM":4,
    "5 ROOM":5,
    "EXECUTIVE":6,
    "MULTI-GENERATION":7
}

combined_df["flat_type"] = combined_df["flat_type"].map(flat_type_map)


# categorise storey range
storey_split = combined_df["storey_range"].str.split(" TO ", expand=True)

combined_df["storey_low"] = storey_split[0].astype(int)
combined_df["storey_high"] = storey_split[1].astype(int)

combined_df["storey_mid"] = (combined_df["storey_low"] + combined_df["storey_high"]) / 2

combined_df["storey_percentile"] = (
    combined_df.groupby("town")["storey_mid"]
    .rank(pct=True)
)

combined_df["storey_relative_category"] = pd.cut(
    combined_df["storey_percentile"],
    bins=[0,0.33,0.66,1],
    labels=["LOW_IN_ESTATE","MID_IN_ESTATE","HIGH_IN_ESTATE"]
)

cols_to_drop = [
    "storey_low",
    "storey_high",
    "storey_range",
    "storey_mid",
    "storey_percentile"
]

combined_df.drop(columns=cols_to_drop, errors="ignore", inplace=True)

print(combined_df.columns)
print(combined_df.head())

Index(['town', 'flat_type', 'block', 'street_name', 'floor_area_sqm',
       'flat_model', 'lease_commence_date', 'resale_price', 'transaction_year',
       'transaction_month', 'remaining_lease_total_months',
       'remaining_lease_years', 'remaining_lease_months', 'flat_age',
       'price_per_sqm', 'log_price', 'log_price_per_sqm', 'mature_estate',
       'storey_relative_category'],
      dtype='object')
         town  flat_type block       street_name  floor_area_sqm  \
0  ANG MO KIO        1.0   309  ANG MO KIO AVE 1            31.0   
1  ANG MO KIO        1.0   309  ANG MO KIO AVE 1            31.0   
2  ANG MO KIO        1.0   309  ANG MO KIO AVE 1            31.0   
3  ANG MO KIO        1.0   309  ANG MO KIO AVE 1            31.0   
4  ANG MO KIO        3.0   216  ANG MO KIO AVE 1            73.0   

       flat_model  lease_commence_date  resale_price  transaction_year  \
0        IMPROVED                 1977        9000.0              1990   
1        IMPROVED             

# Adjust price for inflation (2024 base)

In [12]:
cpi_path = "../data/cpi.xlsx"
cpi_raw = pd.read_excel(cpi_path, sheet_name="T3", header=10)

cpi_raw.head()

,Data Series,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1970,1969,1968,1967,1966,1965,1964,1963,1962,1961
0,All Items,100.903,100.0,97.666,93.163,87.781,85.794,85.942,85.457,85.084,...,23.263,23.196,23.223,23.057,22.345,21.902,21.848,21.494,21.046,20.953
1,Food,101.180,100.0,97.24,91.902,87.266,86.018,84.449,83.185,82.006,...,20.706,20.826,21.183,21.131,20.199,19.648,19.746,19.263,18.579,18.454
2,Food Excl Food & Beverage Serving Services,101.096,100.0,98.79,93.966,89.266,87.865,85.408,84.455,83.367,...,na,na,na,na,na,na,na,na,na,na
3,Rice & Cereal Products,101.621,100.0,96.482,90.274,86.768,86.181,84.505,83.191,82.014,...,na,na,na,na,na,na,na,na,na,na
4,Rice,100.579,100.0,97.842,94.995,96.322,98.206,96.513,92.848,91.589,...,na,na,na,na,na,na,na,na,na,na


In [13]:
cpi_all = cpi_raw[cpi_raw["Data Series"] == "All Items"]
cpi_long = cpi_all.melt(
    id_vars="Data Series",
    var_name="transaction_year",
    value_name="CPI"
)
cpi_long = cpi_long.drop(columns=["Data Series"])
cpi_long["transaction_year"] = cpi_long["transaction_year"].astype(int)

cpi_long.head()


relevant_years = combined_df["transaction_year"].dropna().unique()
cpi_long = cpi_long[cpi_long["transaction_year"].isin(relevant_years)].copy()

# Merge into main dataframe
combined_df = combined_df.merge(cpi_long, on="transaction_year", how="left")

# Check result
print(cpi_long.head())
print(combined_df[["transaction_year", "CPI"]].head())
print("Missing CPI values:", combined_df["CPI"].isna().sum())

   transaction_year      CPI
0              2025  100.903
1              2024    100.0
2              2023   97.666
3              2022   93.163
4              2021   87.781
   transaction_year     CPI
0              1990  53.898
1              1990  53.898
2              1990  53.898
3              1990  53.898
4              1990  53.898
Missing CPI values: 4398


In [14]:
# drop 2026 transactions

missing_years = combined_df.loc[combined_df["CPI"].isna(), "transaction_year"].unique()
print(sorted(missing_years))
combined_df = combined_df[combined_df["transaction_year"] <= 2024].copy()
print("Missing CPI values:", combined_df["CPI"].isna().sum())

[2026]
Missing CPI values: 0


# real_price = resale_price * (CPI_base / CPI_year)

In [15]:
# Get base CPI (2024)
base_cpi = cpi_long.loc[cpi_long["transaction_year"] == 2024, "CPI"].iloc[0]

# Calculate real price
combined_df["real_price"] = (
    combined_df["resale_price"] * base_cpi / combined_df["CPI"]
)

combined_df["real_price"] = pd.to_numeric(combined_df["real_price"], errors="coerce")
combined_df["log_real_price"] = np.log(combined_df["real_price"])

combined_df.head()

,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,transaction_year,transaction_month,...,remaining_lease_months,flat_age,price_per_sqm,log_price,log_price_per_sqm,mature_estate,storey_relative_category,CPI,real_price,log_real_price
0,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,9000.0,1990,1,...,0,13,290.322581,9.104980,5.670993,1,HIGH_IN_ESTATE,53.898,16698.207726,9.723057
1,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,6000.0,1990,1,...,0,13,193.548387,8.699515,5.265528,1,LOW_IN_ESTATE,53.898,11132.138484,9.317592
2,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,8000.0,1990,1,...,0,13,258.064516,8.987197,5.553210,1,HIGH_IN_ESTATE,53.898,14842.851312,9.605274
3,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,6000.0,1990,1,...,0,13,193.548387,8.699515,5.265528,1,MID_IN_ESTATE,53.898,11132.138484,9.317592
4,ANG MO KIO,3.0,216,ANG MO KIO AVE 1,73.0,NEW GENERATION,1976,47200.0,1990,1,...,0,14,646.575342,10.762149,6.471690,1,LOW_IN_ESTATE,53.898,87572.822739,11.380226


In [17]:
# Create the full_address column
combined_df['full_address'] = combined_df['block'] + ' ' + combined_df['street_name']

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

def geocode_onemap(search_val, session=None, auth_token=None, retries=3, sleep=0.3):
    s = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for i in range(retries):
        try:
            r = s.get(SEARCH_URL, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            j = r.json()
            if int(j.get("found", 0)) > 0:
                best = j["results"][0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
            return None, None
        except Exception:
            if i == retries - 1:
                return None, None
            time.sleep((i + 1) * sleep)

# Cache unique addresses
addr_unique = combined_df["full_address"].dropna().unique()
addr_map = {}
sess = requests.Session()

for a in addr_unique:
    addr_map[a] = geocode_onemap(a, session=sess)

combined_df["lat"] = combined_df["full_address"].map(lambda x: addr_map.get(x, (None, None))[0])
combined_df["lon"] = combined_df["full_address"].map(lambda x: addr_map.get(x, (None, None))[1])
combined_df.head()

,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,transaction_year,transaction_month,...,log_price,log_price_per_sqm,mature_estate,storey_relative_category,CPI,real_price,log_real_price,full_address,lat,lon
0,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,9000.0,1990,1,...,9.104980,5.670993,1,HIGH_IN_ESTATE,53.898,16698.207726,9.723057,309 ANG MO KIO AVE 1,1.377017,103.834733
1,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,6000.0,1990,1,...,8.699515,5.265528,1,LOW_IN_ESTATE,53.898,11132.138484,9.317592,309 ANG MO KIO AVE 1,1.377017,103.834733
2,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,8000.0,1990,1,...,8.987197,5.553210,1,HIGH_IN_ESTATE,53.898,14842.851312,9.605274,309 ANG MO KIO AVE 1,1.377017,103.834733
3,ANG MO KIO,1.0,309,ANG MO KIO AVE 1,31.0,IMPROVED,1977,6000.0,1990,1,...,8.699515,5.265528,1,MID_IN_ESTATE,53.898,11132.138484,9.317592,309 ANG MO KIO AVE 1,1.377017,103.834733
4,ANG MO KIO,3.0,216,ANG MO KIO AVE 1,73.0,NEW GENERATION,1976,47200.0,1990,1,...,10.762149,6.471690,1,LOW_IN_ESTATE,53.898,87572.822739,11.380226,216 ANG MO KIO AVE 1,1.366197,103.841505
